# KALA — Finetuning Multimodal Product Search (`CLIP ViT-B/32`)
Notebook ini mengadaptasi model `sentence-transformers/clip-ViT-B-32` pada katalog produk fashion menggunakan `SentenceTransformerTrainer` dan `MultipleNegativesRankingLoss`.

In [ ]:
!pip install -q -U "sentence-transformers[image,train]" datasets accelerate matplotlib pillow

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss

print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 1. Muat Dataset Katalog Fashion
raw_stream = load_dataset('benitomartin/fashion-product-images-small-384x512', split='train', streaming=True)

samples = []
for row in raw_stream:
    title = row.get('productDisplayName')
    img = row.get('image')
    if title and img is not None:
        samples.append({'anchor': str(title), 'positive': img.convert('RGB')})
    if len(samples) >= 600:
        break

ds = Dataset.from_list(samples).train_test_split(test_size=120, seed=42)
train_dataset, eval_dataset = ds['train'], ds['test']
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# 2. Evaluasi Base Model CLIP ViT-B/32
model = SentenceTransformer('sentence-transformers/clip-ViT-B-32')

def evaluate_ndcg10(m, test_ds):
    q_emb = m.encode([x['anchor'] for x in test_ds], batch_size=16, normalize_embeddings=True)
    d_emb = m.encode([x['positive'] for x in test_ds], batch_size=16, normalize_embeddings=True)
    sims = np.matmul(q_emb, d_emb.T)
    scores = []
    for i in range(len(test_ds)):
        top10 = np.argsort(-sims[i])[:10]
        dcg = next((1.0 / math.log2(r + 2) for r, idx in enumerate(top10) if idx == i), 0.0)
        scores.append(dcg)
    return float(np.mean(scores))

base_score = evaluate_ndcg10(model, eval_dataset)
print(f'Base CLIP ViT-B/32 NDCG@10 = {base_score:.4f}')

In [ ]:
# 3. Finetuning dengan MultipleNegativesRankingLoss
loss = MultipleNegativesRankingLoss(model=model, scale=20.0)

args = SentenceTransformerTrainingArguments(
    output_dir='clip-ecommerce-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy='epoch',
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)
trainer.train()

finetuned_score = evaluate_ndcg10(model, eval_dataset)
print(f'Finetuned CLIP ViT-B/32 NDCG@10 = {finetuned_score:.4f}')
model.save_pretrained('clip-ecommerce-finetuned')

In [ ]:
# 4. Plot Grafik Perbandingan NDCG@10
plt.figure(figsize=(7, 4.5), dpi=140)
bars = plt.bar(['Base Model\n(CLIP ViT-B/32)', 'Finetuned Model\n(KALA E-Commerce)'], [base_score, finetuned_score], color=['#64748b', '#111111'], width=0.45)
plt.ylim(0, 1.05)
plt.ylabel('NDCG@10')
plt.title('Multimodal Product Search Benchmark (NDCG@10)')
for b, v in zip(bars, [base_score, finetuned_score]):
    plt.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark_ndcg10.png')
plt.show()